# IterativeFib (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/IterativeFib_python.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `IterativeFib`: l'algoritmo in Python, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/IterativeFib_cpp.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/IterativeFib_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_python.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Python 3, senza installare nulla.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno i nomi dello pseudocodice. Due eccezioni in Python, dove `from` e `next` hanno un significato speciale: `Hanoi` usa `frm` al posto di `from` e `IterativeFib` usa `nxt` al posto di `next`.

## Strumenti: contatori, stack e traccia

La cella definisce i contatori (`cnt`, `calls`, `max_depth`), lo stack delle chiamate (`push` e `pop`) e le funzioni che stampano la traccia (`event`, `say`) e la tabella dei passi (`print_steps`). Basta eseguirla una volta.

In [1]:
cnt = [0] * 20      # cnt[r] = numero di esecuzioni della riga r (righe numerate come nelle dispense)
calls = 0           # numero di chiamate = nodi dell'albero delle chiamate
max_depth = 0       # profondita' della ricorsione = altezza massima dello stack
mults = 0           # moltiplicazioni (FastPower)
moves = 0           # mosse (Hanoi)
ev = 0              # contatore degli eventi della traccia
stack = []          # lo stack delle chiamate: gli argomenti delle chiamate attive
seen = [0] * 64     # seen[x] = numero di chiamate con argomento x
trace = False       # True: stampa la traccia dell'esecuzione


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # inizia una chiamata: il suo frame va in cima allo stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # la chiamata termina: il suo frame esce dallo stack
    stack.pop()


def event(text):                   # riga numerata della traccia, con lo stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Riga | Esecuzioni")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Passi totali S = %d" % sum(cnt))

## Fibonacci iterativo: IterativeFib

**Input:** un numero naturale $n$. **Output:** $F(n)$, calcolato da $F(0)$ e $F(1)$ in su tenendo solo gli ultimi due valori. $S(n) = 5n + 1 = \Theta(n)$ passi. Invariante: prima di ogni valutazione della condizione, $prev = F(i-2)$ e $curr = F(i-1)$.

Pseudocodice: sezione 14.5.2 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
def iterative_fib(n, trace=False):
    cnt[2] += 1
    if n <= 1:
        cnt[3] += 1
        return n
    prev = 0; cnt[5] += 1
    curr = 1; cnt[6] += 1
    i = 2; cnt[7] += 1
    if trace:                                      # traccia: riga di inizializzazione
        print("%-10s | %-8s | %-18s | %4d | %4d" % ("inizio", "2", "-", prev, curr))
    cnt[8] += 1
    while i <= n:
        nxt = prev + curr; cnt[9] += 1             # next nello pseudocodice (in Python e' una funzione predefinita)
        prev = curr; cnt[10] += 1
        curr = nxt; cnt[11] += 1
        i = i + 1; cnt[12] += 1
        cnt[8] += 1
        if trace:                                  # traccia: i - 2 e' il numero dell'iterazione
            print("%-10d | %-8s | %-18s | %4d | %4d" % (i - 2, "%d -> %d" % (i - 1, i), "%d + %d = %d" % (nxt - prev, prev, nxt), prev, curr))
    cnt[14] += 1
    return curr

In [3]:
# Esempio delle dispense: IterativeFib(4)
n = 4
reset()
print("iterazione | i        | next = prev + curr | prev | curr")
r = iterative_fib(n, True)
print("Il ciclo termina con i = %d > %d: restituisce curr = %d" % (n + 1, n, r))
print_steps(2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14)
print("S(n) = 5n + 1 = %d" % (5 * n + 1))

# Confronto con Fib: passi di Fib = 4F(n+1) - 2 (formula verificata nella sezione precedente)
print()
print("  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1")
for n in [5, 10, 20, 30]:
    reset()
    r = iterative_fib(n)
    steps = sum(cnt)
    print("%3d | %6d | %16d | %12d | %d" % (n, r, 4 * iterative_fib(n + 1) - 2, steps, 5 * n + 1))

iterazione | i        | next = prev + curr | prev | curr
inizio     | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
Il ciclo termina con i = 5 > 4: restituisce curr = 3
Riga | Esecuzioni
   2 |          1
   3 |          0
   5 |          1
   6 |          1
   7 |          1
   8 |          4
   9 |          3
  10 |          3
  11 |          3
  12 |          3
  14 |          1
Passi totali S = 21
S(n) = 5n + 1 = 21

  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1
  5 |      5 |               30 |           26 | 26
 10 |     55 |              354 |           51 | 51
 20 |   6765 |            43782 |          101 | 101
 30 | 832040 |          5385074 |          151 | 151


**Prova tu.** Cambia $n$ e controlla che i passi siano $5n + 1$.

In [4]:
n = 10        # <-- cambia n
reset()
print("iterazione | i        | next = prev + curr | prev | curr")
r = iterative_fib(n, True)
print("F(%d) = %d, S = %d = 5n + 1 = %d" % (n, r, sum(cnt), 5 * n + 1))

iterazione | i        | next = prev + curr | prev | curr
inizio     | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
4          | 5 -> 6   | 2 + 3 = 5          |    3 |    5
5          | 6 -> 7   | 3 + 5 = 8          |    5 |    8
6          | 7 -> 8   | 5 + 8 = 13         |    8 |   13
7          | 8 -> 9   | 8 + 13 = 21        |   13 |   21
8          | 9 -> 10  | 13 + 21 = 34       |   21 |   34
9          | 10 -> 11 | 21 + 34 = 55       |   34 |   55
F(10) = 55, S = 51 = 5n + 1 = 51


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).